# Phase 6 — Test the Repetition-Guard Mechanism
Tests GuardedChainProgram (repetition guard + malformed-JSON retry) on Tier 3 (where DSPy got 0/6) and Tier 2 (where DSPy already got 6/6, to confirm the guard doesn't break anything that already worked).

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: must show a Tesla T4 GPU table with 0MiB used.**

## 1. Clone repo (safe to re-run any time)

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pwd

In [ ]:
!grep -c "class GuardedChainProgram" dspy_optimize_tier2.py

Must print `1`. If it prints `0`, the guard code wasn't pushed correctly — check before continuing.

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets dspy-ai optuna

In [ ]:
from huggingface_hub import login
login()

## 2. Load model

In [ ]:
import sys, json
sys.path.insert(0, ".")
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from envs.training_data import sample_tier3_training, sample_tier2_training
from tasks.tier3 import TIER3_HELDOUT
from tasks.tier2 import TIER2_HELDOUT
import dspy_optimize_tier3, dspy_optimize_tier2

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model + DSPy LM wrapper ready.")

## 3. Tier 3 with the guard — the main test
This is the run that matters: does the guard fix the 0/6 result?

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

train_10_t3 = sample_tier3_training(10)
print(f"Training on {len(train_10_t3)} Tier 3 examples.")

optimized_guarded_t3 = dspy_optimize_tier3.optimize(lm, train_10_t3, guarded=True)
print("Guarded DSPy optimization (Tier 3) complete.")

In [ ]:
guarded_results_t3 = dspy_optimize_tier3.evaluate_program(optimized_guarded_t3, TIER3_HELDOUT)
for r in guarded_results_t3:
    print(f"[{'PASS' if r['grade']['success'] else 'FAIL'}] {r['id']} — {r['grade'].get('failure_type')}")

guarded_success_rate_t3 = sum(r["grade"]["success"] for r in guarded_results_t3) / len(guarded_results_t3)
print(f"\nGuarded DSPy Tier 3 (N=10) held-out success rate: {guarded_success_rate_t3:.1%}")
print("(Compare to un-guarded result: 0% (0/6))")

with open("results/tier3_dspy_guarded_n10_results.json", "w") as f:
    json.dump(guarded_results_t3, f, indent=2)

## 4. Download Tier 3 guarded results NOW

In [ ]:
from google.colab import files
files.download("results/tier3_dspy_guarded_n10_results.json")

## 5. Tier 2 with the guard — regression check
Tier 2 already got 6/6 without the guard. This confirms the guard doesn't break what already worked. Same session is fine here (no model reload needed), but if you hit memory pressure, restart first.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

train_10_t2 = sample_tier2_training(10)
print(f"Training on {len(train_10_t2)} Tier 2 examples.")

optimized_guarded_t2 = dspy_optimize_tier2.optimize(lm, train_10_t2)
print("Note: dspy_optimize_tier2.optimize() doesn't take a guarded= flag; ")
print("this cell manually builds the guarded program below instead.")

The above cell's `optimize()` call still uses the plain ChainProgram internally (Tier 2's `optimize()` wasn't given a `guarded=` flag). To actually test the guard on Tier 2, build and optimize a GuardedChainProgram directly:

In [ ]:
import dspy
gc.collect()
torch.cuda.empty_cache()

dspy.settings.configure(lm=lm)
lm.max_new_tokens = 150
guarded_program_t2 = dspy_optimize_tier2.GuardedChainProgram(max_turns=6)
trainset_t2 = dspy_optimize_tier2.build_trainset(train_10_t2)

optimizer_t2 = dspy.MIPROv2(metric=dspy_optimize_tier2.tier2_metric, auto=None, num_candidates=1, num_threads=1)
optimized_guarded_t2 = optimizer_t2.compile(
    guarded_program_t2, trainset=trainset_t2,
    num_trials=3, max_bootstrapped_demos=1, max_labeled_demos=1,
    minibatch=False, requires_permission_to_run=False,
    program_aware_proposer=False, data_aware_proposer=False,
    tip_aware_proposer=False, fewshot_aware_proposer=False,
)
print("Guarded DSPy optimization (Tier 2) complete.")

In [ ]:
guarded_results_t2 = dspy_optimize_tier2.evaluate_program(optimized_guarded_t2, TIER2_HELDOUT)
for r in guarded_results_t2:
    print(f"[{'PASS' if r['grade']['success'] else 'FAIL'}] {r['id']} — {r['grade'].get('failure_type')}")

guarded_success_rate_t2 = sum(r["grade"]["success"] for r in guarded_results_t2) / len(guarded_results_t2)
print(f"\nGuarded DSPy Tier 2 (N=10) held-out success rate: {guarded_success_rate_t2:.1%}")
print("(Compare to un-guarded result: 100% (6/6) — this should still be 100%, confirming no regression)")

with open("results/tier2_dspy_guarded_n10_results.json", "w") as f:
    json.dump(guarded_results_t2, f, indent=2)

## 6. Download everything and push

In [ ]:
from google.colab import files
files.download("results/tier2_dspy_guarded_n10_results.json")

Move both downloaded files into `results/` on your laptop, then:
```bash
git add results/tier3_dspy_guarded_n10_results.json results/tier2_dspy_guarded_n10_results.json
git commit -m "Phase 6: repetition-guard results for Tier 2 and Tier 3"
git push
```